# Hate Speech Detection

In [1]:
import pandas as pd
import re

In [2]:
train = pd.read_csv("train.csv")

In [3]:
test = pd.read_csv("test.csv")

In [4]:
def cleaning_text(df, text_field):
    df[text_field] = df[text_field].str.lower()
    df[text_field] = df[text_field].apply(lambda elem: re.sub(r"(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|^rt|http.+?", "", elem))
    return df

In [5]:
test_clean = cleaning_text(test, "tweet")
train_clean = cleaning_text(train, "tweet")

In [9]:
from sklearn.utils import resample
train_major = train_clean[train_clean.label == 0]
train_minor = train_clean[train_clean.label == 1]
train_minor_upsampled = resample(train_minor, replace = True, n_samples = len(train_major), random_state = 123)
train_upsampled = pd.concat([train_minor_upsampled, train_major])
train_upsampled["label"].value_counts()

label
1    29720
0    29720
Name: count, dtype: int64

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier

In [11]:
pipeline_SGD = Pipeline([("vect", CountVectorizer()), ("tfidf", TfidfTransformer()), ("nb", SGDClassifier()), ])

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(train_upsampled["tweet"], train_upsampled["label"], random_state = 0)

In [14]:
model = pipeline_SGD.fit(X_train, y_train)
y_predict = model.predict(X_test)

In [17]:
from sklearn.metrics import f1_score
f1_score(y_test, y_predict)

0.969729297239632